# 01 — SI Preprocessing

Scans all MEA recordings under `DATA_ROOT`, registers a single pipeline task
**`SI-Preprocessing`**, and runs SpikeInterface preprocessing on every well:

1. Unsigned → signed conversion (when needed)
2. Bandpass filter 300–3000 Hz
3. Local common median reference (fallback to global if electrode locations are missing)
4. Save preprocessed recording as a **zarr** compressed file

The pipeline cache makes re-runs safe: already-complete wells are skipped automatically.
Failed wells can be reset with `pipeline_mgr.refresh("SI-Preprocessing", ...)`.

In [ ]:
# ============================================================
# ⚙️  CONFIG — edit this cell before running
# ============================================================
from pathlib import Path

# --- Paths ---
DATA_ROOT                 = Path("/Volumes/MEA_Backup/raw_data/rbs_maxtwo_desktop/harddisk20tbV2/Sadegh_Lab/CX138")        # root of MEA raw recordings
ANALYSIS_ROOT             = Path("../data/test_data/analysis_output")         # dataset + pipeline cache
PREPROCESSING_OUTPUT_ROOT = Path("../data/test_data/analysis_output/preprocessing_output")    # zarr output root

# --- SpikeInterface compute resources ---
N_JOBS         = 4       # parallel workers for saving
CHUNK_DURATION = "1s"    # chunk size fed to SI workers

# --- Recording selection ---
RECORDING_NUM = "rec0000"  # HDF5 recording name inside each run file

# --- Preprocessing parameters ---
BANDPASS_FREQ_MIN   = 300      # Hz — lower edge
BANDPASS_FREQ_MAX   = 3000     # Hz — upper edge
CMR_LOCAL_RADIUS_UM = (0, 250) # (inner_um, outer_um) for local CMR annulus
# ============================================================

In [ ]:
import sys
import traceback
import logging

import pandas as pd
import spikeinterface.full as si
import spikeinterface.preprocessing as spre

# Allow importing from the repo root when running from notebooks/
_repo_root = str(Path("..").resolve())
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

from dataset_manager import DatasetManager
from pipeline_manager import PipelineManager
from pipeline_manager.task_record import TaskStatus

logging.basicConfig(level=logging.WARNING)  # suppress SI/DatasetManager chatter
print("Imports OK")

## 1. Scan recordings

In [ ]:
ANALYSIS_ROOT.mkdir(parents=True, exist_ok=True)

dataset_mgr = DatasetManager(DATA_ROOT, ANALYSIS_ROOT)
recordings  = dataset_mgr.recordings

print(f"Found {len(recordings)} recording(s)")

summary_rows = [
    {
        "recording_key": r.cache_key,
        "date":          r.date,
        "sample_id":     r.sample_id,
        "plate_id":      r.plate_id,
        "scan_type":     r.scan_type,
        "run_id":        r.run_id,
        "n_wells":       len(r.wells),
        "size_GB":       round(r.file_size / 1e9, 2),
    }
    for r in recordings
]
pd.DataFrame(summary_rows)

## 2. Register pipeline task and add wells

In [ ]:
TASK_NAME = "SI-Preprocessing"

pipeline_mgr = PipelineManager(ANALYSIS_ROOT)

# register_computation_task raises ValueError on duplicate — safe to re-run
try:
    pipeline_mgr.register_computation_task(TASK_NAME, dependencies=[])
    print(f"Registered task: {TASK_NAME!r}")
except ValueError as e:
    print(f"Task already registered ({e}) — continuing")

n_added = 0
n_skipped_no_wells = 0
for rec in recordings:
    if not rec.wells:
        print(f"WARNING: no wells found for {rec.cache_key!r} — skipping")
        n_skipped_no_wells += 1
        continue
    for well_id in rec.wells:
        pipeline_mgr.add_well(rec.cache_key, well_id)
        n_added += 1

print(f"\nWells registered: {n_added}  |  Recordings skipped (no wells): {n_skipped_no_wells}")

## 3. Pipeline status overview

In [ ]:
def _status_table(mgr: PipelineManager, task_name: str) -> pd.DataFrame:
    rows = []
    for entry in mgr.entries:
        rec = entry.tasks.get(task_name)
        rows.append({
            "recording_key": entry.recording_key,
            "well_id":        entry.well_id,
            "status":         rec.status if rec else "—",
            "output_path":    str(rec.output_path) if rec and rec.output_path else "",
            "error":          (rec.error or "")[:120] if rec else "",
        })
    return pd.DataFrame(rows)


df_status = _status_table(pipeline_mgr, TASK_NAME)
print(df_status["status"].value_counts().to_string())
df_status

## 4. Preprocessing function

In [ ]:
def run_si_preprocessing(
    data_path: Path,
    stream_id: str,
    output_path: Path,
) -> Path:
    """Preprocess one well and save as zarr. Returns the zarr folder path."""

    # 1. Load
    rec = si.read_maxwell(
        str(data_path),
        stream_id=stream_id,
        rec_name=RECORDING_NUM,
    )

    # 2. Unsigned → signed
    if rec.get_dtype().kind == "u":
        rec = spre.unsigned_to_signed(rec)

    # 3. Bandpass filter
    rec = spre.bandpass_filter(
        rec,
        freq_min=BANDPASS_FREQ_MIN,
        freq_max=BANDPASS_FREQ_MAX,
    )

    # 4. Local common median reference; fall back to global when locations are absent
    try:
        rec = spre.common_reference(
            rec,
            reference="local",
            operator="median",
            local_radius=CMR_LOCAL_RADIUS_UM,
        )
    except Exception as _cmr_err:
        print(f"    Local CMR failed ({_cmr_err}), falling back to global CMR")
        rec = spre.common_reference(rec, reference="global", operator="median")

    rec.annotate(is_filtered=True)

    # 5. Save as zarr (default Blosc/LZ4 compression via SpikeInterface)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    rec.save(
        folder=output_path,
        format="zarr",
        overwrite=True,
        n_jobs=N_JOBS,
        chunk_duration=CHUNK_DURATION,
        progress_bar=True,
    )
    return output_path


print("Preprocessing function defined")

## 5. Run preprocessing

Already-complete wells are skipped automatically.  
To re-run a failed well: `pipeline_mgr.refresh(TASK_NAME, recording_key=..., well_id=...)`  
To reset everything: `pipeline_mgr.refresh(TASK_NAME)`

In [ ]:
# Build a lookup dict for fast recording resolution
_rec_lookup = {r.cache_key: r for r in recordings}

while True:
    work_items = pipeline_mgr.get_next_task(n=1)
    if not work_items:
        break

    item      = work_items[0]
    rec_entry = _rec_lookup[item.recording_key]

    zarr_path = (
        PREPROCESSING_OUTPUT_ROOT
        / item.recording_key   # e.g. SampleA/240415/plate1/scan1/001
        / item.well_id         # e.g. well000
        / "preprocessed.zarr"
    )

    print(f"\nProcessing: {item.recording_key} / {item.well_id}")
    pipeline_mgr.update_status(item, TaskStatus.RUNNING)

    try:
        run_si_preprocessing(rec_entry.data_path, item.well_id, zarr_path)
        pipeline_mgr.update_status(item, TaskStatus.COMPLETE, output_path=zarr_path)
        print(f"  ✓  saved → {zarr_path}")
    except Exception as exc:
        pipeline_mgr.update_status(item, TaskStatus.FAILED, error=str(exc))
        traceback.print_exc()
        print(f"  ✗  failed: {exc}")

print("\nDone — no more pending tasks.")

## 6. Final status report

In [ ]:
df_final = _status_table(pipeline_mgr, TASK_NAME)

print("=== Summary ===")
print(df_final["status"].value_counts().to_string())

failed = df_final[df_final["status"] == TaskStatus.FAILED]
if not failed.empty:
    print("\n=== Failed wells ===")
    for _, row in failed.iterrows():
        print(f"  {row['recording_key']} / {row['well_id']}")
        print(f"    error: {row['error']}")

df_final